# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [4]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - acb29783


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [6]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [7]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [8]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_3", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
🔄 [todo_3] Research sleep improvement strategies (in_progress)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [9]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: d:\SOFTWARES\CodeRepo\AIE9\07_Deep_Agents\workspace


In [10]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] .gitkeep (0 bytes)
[DIR] research


In [13]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] sleep_notes.md (242 bytes)


In [14]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [15]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: d:\SOFTWARES\CodeRepo\AIE9\07_Deep_Agents\workspace


In [16]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Perfect! I've created a comprehensive, personalized sleep improvement plan that directly addresses your specific challenges. Here's what I've developed for you:

## Your Personalized Sleep Improvement Plan Summary

**The plan is structured in 3 phases over 8 weeks:**

1. **Phase 1 (Weeks 1-2): Foundation Building**
   - Establishing consistent sleep-wake times (targeting 10:30pm bedtime)
   - Removing phone from bedroom and creating digital boundaries
   - Optimizing your sleep environment

2. **Phase 2 (Weeks 3-4): Circadian Rhythm Regulation**
   - Strategic morning light exposure to combat fatigue
   - Creating a proper 60-90 minute wind-down routine
   - Evening light management

3. **Phase 3 (Weeks 5-8): Optimization & Habit Reinforcement**
   - Fine-tuning nutrition, exercise, and stress management
   - Advanced morning energy enhancement techniques
   - Long-term habit solidification

**Key features of your plan:**
- ✅ Directly addresses your inconsistent bedtime

In [18]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep improvement strategies (completed)
✅ [todo_5] Research sleep hygiene best practices (completed)
✅ [todo_7] Save the plan to a file (completed)
✅ [todo_6] Research circadian rhythm regulation techniques (completed)
✅ [todo_8] Research digital devices and blue light impact (completed)
✅ [todo_10] Research bedtime routine establishment strategies (completed)
✅ [todo_12] Research morning alertness and fatigue reduction methods (completed)
✅ [todo_14] Research strategies for inconsistent sleep schedules (completed)
✅ [todo_16] Compile comprehensive research summary (completed)
✅ [todo_18] Format as personalized sleep plan framework (completed)


Workspace contents:
  [FILE] personalized_sleep_improvement_plan.md (6175 bytes)
  [DIR] research/


---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
The todo list is a "no-op" tool, it doesn not execute actions but provides context engineering that helps the agent maintain focus across long tasks. It basically breaks down a complex task into smaller steps and tracks progress.

Trade-Offs of using ToDo lists for planning

1. Explicit planning can slow things down.
Todo lists add an extra step before the agent starts doing the work.
	- The agents spends time writing the list of steps instead of jumping straight into execution.
	- If the task is simple and straightforward like say summarize this paragraph, this planning step might take as long than just doing the task, so in such cases it just adds overhead. This extra planning can add latency and waste tokens.
Planning adds value only when the task's complexity requires it.


2. Choosing the right level of detail matters.
If the ToDo steps are too vague like "finish the project" or too detailed which includes tiny steps, it just makes the list noisy and dont meaningfully advance the task, it just slows down the system. Good todos should be actionable steps that move the agent closer to the goal.

3. If todos are never completed, the plan becomes useless and stalls the workflow.
Agents use todo states (pending --> in_progress --> completed) to decide what to do next. If the agents never update thier status, it can stall the workflow. Without completion updates, we lose the progress made and the feedback loops, the agent does not learn what is done versus what is pending.

Planning via ToDo lists should be justified by the task complexity, use it for multi-step complex tasks and skip it for simple single-turn interaction. For shallow tasks, explict overhead slows the agent down, for deep tasks, planning helps to drive the execution.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
1. What goes in files/external storage
	- Store historical information, large health document, research infor as a file or in vector DB. Retrieve on demand only the relevant sections by creating an index or adding metadata support for different categories. Dont paste the whole document into the system prompt.
	- For user metrics, store the historical data in files, keep only the recent summary in active context and load the historical metrics on demand.
	
2. What goes in the prompt?
	- Safety, critical user conditions like allergies, medications, chronic conditions etc so that model does not miss this information. Safety facts should not depend on retrieval.
	- User preferences
	
3. What should never be offloaded?
	- Anything that could harm the user if missed, in this case allergies, medical condition etc, information that must be checked before every action.
	- Active session state
	- User identity for authentication

4. Handling context limits. Build deep Agents such that they automatically handle context limits:
    - **Large result offloading**: Tool results >20k tokens → saved to disk
    - **Proactive offloading**: At 85% context capacity → agent saves state to disk
    - **Summarization**: Long conversations get summarized while preserving intent

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [ ]:
### YOUR CODE HERE ###

# Step 1: Create a research agent with appropriate tools
# Hint: You'll need file tools to read the wellness guide

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pathlib import Path

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Step 2: Add a tool to read from the data folder
# Hint: Use Path("data/HealthWellnessGuide.txt")

@tool
def read_wellness_guide() -> str:
    """Read the HealthWellnessGuide.txt from the data folder.
    
    This tool provides access to the comprehensive wellness guide
    that contains information about exercise, nutrition, sleep, stress management, building healthy habits, common health concerns and lifestyle.
    
    Returns:
        The full contents of the HealthWellnessGuide.txt file
    """
    guide_path = Path("data/HealthWellnessGuide.txt")
    if not guide_path.exists():
        return f"Error: HealthWellnessGuide.txt not found at {guide_path}"
    
    try:
        content = guide_path.read_text(encoding='utf-8')
        return f"HealthWellnessGuide.txt contents:\n\n{content}"
    except Exception as e:
        return f"Error reading guide: {str(e)}"

# Combine tools: todo tracking + data reading + built-in file tools
research_tools = [
    write_todos,
    update_todo,
    list_todos,
    read_wellness_guide,  # Custom tool to read from data folder
]

# Step 3: Create the agent with a research-focused system prompt
research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=research_tools,
    backend=filesystem_backend,  # File operations go to workspace
    system_prompt="""You are a Wellness Research Agent specialized in conducting thorough research on wellness topics.

## Your Research Process:
1. **Planning**: Always start by creating a todo list to track your research process
2. **Data Gathering**: Use read_wellness_guide() to access the comprehensive wellness guide
3. **Analysis**: Extract relevant information related to the research topic
4. **Synthesis**: Organize findings into a structured, evidence-based report
5. **Documentation**: Save your findings to a well-formatted markdown file in the workspace
6. **Progress Tracking**: Update todo status as you complete each step

## Research Guidelines:
- Focus on evidence-based information from the wellness guide
- Structure your reports with clear sections and headers
- Include specific strategies, techniques, or recommendations
- Cite relevant information when possible
- Create comprehensive guides that are actionable and practical

## Output Format:
- Save research reports as markdown files with descriptive names
- Use proper markdown formatting (headers, lists, emphasis)
- Include an introduction, main content sections, and a summary
- Make the content easy to read and reference

Always update your todos to track progress and ensure all steps are completed."""
)

print("Research Agent created successfully!")
print(f"Workspace: {workspace_path}")

# Step 4: Test with the stress management research task
TODO_STORE.clear()

test_result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."
    }]
})

print("\n" + "="*60)
print("RESEARCH AGENT RESPONSE")
print("="*60)
print(test_result["messages"][-1].content)

# Check todo status
print("\n" + "="*60)
print("TODO STATUS")
print("="*60)
print(list_todos.invoke({}))

# Check generated files
print("\n" + "="*60)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Research Agent created successfully!
Workspace: d:\SOFTWARES\CodeRepo\AIE9\07_Deep_Agents\workspace

RESEARCH AGENT RESPONSE
## Research Complete: Comprehensive Stress Management Guide

I've successfully completed my research on stress management techniques and created a comprehensive evidence-based guide with 7 detailed strategies. Here's what I accomplished:

### **Research Process Completed:**
✅ Planned comprehensive research approach  
✅ Accessed the wellness guide database  
✅ Extracted evidence-based stress management strategies  
✅ Organized findings into a structured, actionable guide  
✅ Created and saved a detailed markdown research report  

### **Key Evidence-Based Strategies Included:**

1. **Deep Breathing Techniques** - Box breathing method for immediate stress relief
2. **Progressive Muscle Relaxation** - Systematic tension/release technique 
3. **Mindfulness and Meditation** - Multiple meditation types with step-by-step instructions
4. **5-4-3-2-1 Grounding Technique**

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [19]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [20]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [21]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created a comprehensive morning routine guide for better energy. Here's what was accomplished:

## ✅ **Project Complete!**

### **Comprehensive Research Phase:**
- **Circadian rhythm science** - How morning routines synchronize your internal clock for better energy
- **Exercise science** - Evidence-based morning workout strategies for all fitness levels
- **Nutrition research** - Optimal breakfast timing, composition, and hydration strategies
- **Mindset practices** - Scientific backing for meditation, gratitude, visualization, and other mental practices

### **Expert Content Creation:**
The writing-agent synthesized all research into a **professional 40+ page guide** that includes:

🧠 **Science-Based Foundation** - Research backing for every recommendation  
💪 **Exercise Section** - Beginner to advanced workout options (20-60 minutes)  
🥗 **Nutrition Guidelines** - Balanced meal formulas and practical breakfast ideas  
🧘 **Mindset Pract

In [22]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Research exercise recommendations for mornings (completed)
✅ [todo_5] Research morning nutrition strategies (completed)
✅ [todo_7] Research mindset and mental practices (completed)
✅ [todo_9] Create comprehensive morning routine guide (completed)
✅ [todo_11] Save guide as formatted markdown file (completed)

Generated files in workspace:
  [FILE] comprehensive_morning_routine_guide.md (18218 bytes)
  [FILE] morning_routine_guide.md (42151 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6175 bytes)
  [DIR] research/
  [FILE] stress_management_research_guide.md (9126 bytes)


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [23]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [24]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [25]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [26]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Alex! Based on your profile, I can see you're focused on improving energy levels and getting better sleep, with a preference for morning workouts. Given your mild anxiety, I'll recommend a routine that's energizing but not overstimulating. Here's a personalized exercise plan for you:

## **Morning Energy-Boosting Routine (4-5 days/week)**

### **Monday, Wednesday, Friday - Active Recovery**
- **5-10 minutes**: Gentle stretching or yoga flow
- **20-25 minutes**: Brisk walk outdoors (natural light helps with sleep regulation)
- **5-10 minutes**: Light bodyweight exercises (squats, modified push-ups, gentle lunges)
- **5 minutes**: Deep breathing or meditation

### **Tuesday, Thursday - Moderate Intensity**
- **5 minutes**: Dynamic warm-up (arm circles, leg swings, gentle twists)
- **15-20 minutes**: Circuit training with 30 seconds work, 30 seconds rest:
  - Bodyweight squats
  - Wall or incline push-ups
  - Walking lunges
  - Standing side bends
  - Marching in place


## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [38]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [28]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [29]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [30]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [31]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Wellness Assessment Summary

Based on your profile as a 35-year-old sedentary office worker wanting to lose 15 pounds while dealing with sleep issues, I've identified **sleep quality as your highest priority**. Here's why: poor sleep directly sabotages weight loss by disrupting hunger hormones and reducing energy for exercise.

### Your Key Strengths:
- Clear, specific goals (15 pounds)
- No major health conditions 
- Already health-conscious with vegetarian diet
- Motivated to make changes

### Critical Areas to Address:
1. **Sleep Quality** (Most Important) - Foundation for everything else
2. **Sedentary Lifestyle** - Breaking up sitting time and adding movement
3. **Weight Loss Strategy** - Vegetarian-friendly approach

## Your Action Plan

### **Start Today:**
1. **Create a bedtime routine** - Same time nightly, no screens 1 hour before bed
2. **Set hourly movement alerts** - Stand and walk 2-3 minutes every hour
3. **Track water intake** - Aim for 8 glasses

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [32]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli not installed. Install with:
  uv pip install deepagents-cli
  # or
  pip install deepagents-cli


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [33]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [34]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [35]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Perfect! I've created your comprehensive 2-week wellness program with all the files you requested. Here's what I've delivered:

## 🎯 Your Complete Wellness Program

I've created **4 detailed reference files** for you:

### 1. **Master Wellness Program** (`/alex_master_wellness_program.md`)
Your complete integrated program with daily schedules, weekly progressions, and long-term sustainability planning.

### 2. **Exercise Plan** (`/alex_exercise_plan.md`)
- 3 distinct 30-minute morning workouts (Full Body, Cardio/Mobility, Core/Stability)
- Monday/Wednesday/Friday schedule
- Progressive difficulty from Week 1 to Week 2
- All exercises designed for your anxiety-friendly approach

### 3. **Vegetarian Nutrition Plan** (`/alex_nutrition_plan.md`)
- Complete 14-day meal plan with recipes
- Pre/post-workout nutrition guidance
- Shopping lists and meal prep strategies
- Foods specifically chosen for energy and sleep support

### 4. **Stress & Sleep Optimization** (`/al

In [36]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create 2-week exercise program (completed)
✅ [todo_3] Develop vegetarian meal plan (completed)
✅ [todo_5] Design stress management and sleep optimization plan (completed)
✅ [todo_7] Create master wellness program document (completed)
✅ [todo_9] Save individual plan files (completed)

GENERATED FILES
  [FILE] alex_exercise_plan.md (2414 bytes)
  [FILE] alex_master_wellness_program.md (7482 bytes)
  [FILE] alex_nutrition_plan.md (4948 bytes)
  [FILE] alex_stress_sleep_management_plan.txt (2517 bytes)
  [FILE] alex_stress_sleep_plan.md (6718 bytes)
  [FILE] comprehensive_morning_routine_guide.md (18218 bytes)
  [FILE] morning_routine_guide.md (42151 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6175 bytes)
  [DIR] research/
  [FILE] stress_management_research_guide.md (9126 bytes)
  [DIR] workspace/


In [37]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of alex_exercise_plan.md:
# Alex's 2-Week Exercise Program

## Overview
A comprehensive 30-minute morning exercise routine designed to improve energy levels, build consistency, and support better sleep while being mindful of mild anxiety.

## Weekly Schedule
- **Week 1 & 2:**
  - Monday: Workout A (Full Body Strength)
  - Wednesday: Workout B (Cardio and Mobility)  
  - Friday: Workout C (Core and Stability)

## Workout Details

### Warm-Up (5 minutes for all workouts)
1. **March in Place** - 2 minutes
2. **Arm Circles** - 1 minute (30 seconds forward, 30 seconds backward)
3. **Leg Swings** - 1 minute (30 seconds each leg)
4. **Dynamic Stretching** - 1 minute (side lunges)

### Workout A - Full Body Strength (Monday)
- **Bodyweight Squats**: 2 sets of 10-12 reps
- **Modified Push-Ups** (knees on ground): 2 sets of 8-10 reps
- **Standing Dumbbell Shoulder Press** (light weights): 2 sets of 10 reps
- **Plank**: Hold for 20-30 seconds
- **Cool Down** (5 minutes): Stretch focusin

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
Key considerations for Subagent configurations:
1. Share tools when subagents need the same capabilities like file read/write/write_todos/update_todos/list_todos etc. Use distinct tools when tasks differ and each subagent has its own expertise/specialization. The key here is to share common tools and give specialized tools only to the subagents that need them.
2. Use cheaper models like gpt-4o-mini for straightforward simple tasks like research, data gethering, for sub agents. Use more capable models like calude-sonnet-4 for complex tasks like synthesis, decision making, for deep agents, supervisor agents. Match model features, cost to impact.
3. One subagent per distinct domain like research, writing, exercise planning, nutrition planning etc without overlapping the responsibilities and each having a clear role. One subagent doing everything defeats the purpose and too many subagents adds overhead and coordination cost.

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
1. Replace in-memory stores with a PostgreSQL database. Store user profiles, preferences, history so that data persists across restarts. Use the FilesystemBackend for file operations.
2. Authentication and authorization for multi-user support. Isolate each user's data and query using user_id and respective namespaces.
3. Using LangSmith for monitoring and observability. Log all agent actions with reasoning. Track all required metrics like latency, error rates, token usage, costs etc. Set up alerts for failures and unusual patterns, errors, exceptions.
4. Track api costs associated with each sub-agent. Set budgets per user/session. Use appropriate models based on the complexity and features. Add features like rate limiting to prevent API abuse, caching for quick responses, streaming response for better user experience.
5. Safety guardrails a must for health advice to validate responses to flag any medical claims or advices. Add required disclaimers. Add HITL when required.

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [14]:
### YOUR CODE HERE ###

# NOTE - Run cell 1 to 4 for API keys
# TODO tools cell to load common planning tools write_todos, update_todo, list_todos
# Longterm memory aware tools - get_user_profile, save_user_preference
# Skills.md tool - load_skill

#Imports
from pathlib import Path
from typing import Literal, List, Dict, Any
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.store.memory import InMemoryStore


# Step 1: Define your subagent configurations

# Create a dedicated filesystem backend for Activity #2
activity_two_workspace = Path("workspace/activity_two").absolute()
activity_two_workspace.mkdir(parents=True, exist_ok=True)

# Create a separate FilesystemBackend specifically for Activity #2
activity_two_backend = FilesystemBackend(
    root_dir=str(activity_two_workspace),
    virtual_mode=True  # Required to sandbox file operations!
)

print(f"Activity #2 workspace configured: {activity_two_workspace}")

# Challenge Planning Specialist - Creates structured 30-day plans
challenge_planner = {
    "name": "challenge-planner",
    "description": "Expert in creating structured wellness challenge plans. Use for designing 30-day programs with daily activities and milestones.",
    "system_prompt": """You are a wellness challenge planning specialist. Your expertise includes:
- Designing progressive 30-day wellness programs
- Breaking down goals into daily actionable steps
- Creating milestone checkpoints
- Balancing challenge difficulty with achievability

When creating plans:
- Start with foundational activities in Week 1
- Gradually increase intensity/duration
- Include rest days and recovery periods
- Make activities specific and measurable
- Consider user's current fitness level and constraints

Output well-structured markdown plans with clear daily activities.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

# Progress Tracking Specialist - Analyzes progress and adapts recommendations
progress_analyst = {
    "name": "progress-analyst",
    "description": "Expert in analyzing wellness progress and adapting recommendations. Use for reviewing daily check-ins and adjusting plans.",
    "system_prompt": """You are a progress tracking and adaptation specialist. Your expertise includes:
- Analyzing daily check-in data
- Identifying patterns in progress
- Recognizing barriers and challenges
- Adapting plans based on feedback
- Creating weekly summary reports

When analyzing progress:
- Look for trends (improving, declining, stagnant)
- Identify what's working well
- Spot potential issues early
- Suggest realistic adjustments
- Celebrate wins and provide encouragement

Create clear, actionable weekly summaries with specific recommendations.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Subagent configurations defined!")


# Step 2: Create any additional tools you need
@tool
def save_daily_checkin(user_id: str, day: int, data: dict) -> str:
    """Save a daily check-in entry for the 30-day challenge.
    
    Args:
        user_id: The user's unique identifier
        day: Day number in the challenge (1-30)
        data: Dictionary with check-in data (e.g., {"energy": 7, "completed_activities": ["exercise", "meditation"], "notes": "..."})
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "daily_checkins")
    memory_store.put(namespace, f"day_{day}", data)
    return f"Saved check-in for day {day}"

@tool
def get_daily_checkin(user_id: str, day: int) -> str:
    """Retrieve a specific daily check-in entry.
    
    Args:
        user_id: The user's unique identifier
        day: Day number to retrieve
    
    Returns:
        Formatted check-in data or "Not found" message
    """
    namespace = (user_id, "daily_checkins")
    items = list(memory_store.search(namespace))
    
    for item in items:
        if item.key == f"day_{day}":
            data = item.value
            result = [f"Day {day} Check-in:"]
            for key, value in data.items():
                result.append(f"  {key}: {value}")
            return "\n".join(result)
    
    return f"No check-in found for day {day}"

@tool
def get_weekly_progress(user_id: str, week: int) -> str:
    """Get all check-ins for a specific week (days 1-7, 8-14, 15-21, 22-28, or 29-30).
    
    Args:
        user_id: The user's unique identifier
        week: Week number (1-5)
    
    Returns:
        Summary of all check-ins for that week
    """
    namespace = (user_id, "daily_checkins")
    items = list(memory_store.search(namespace))
    
    # Calculate day range for the week
    start_day = (week - 1) * 7 + 1
    end_day = min(week * 7, 30)
    
    week_data = []
    for day in range(start_day, end_day + 1):
        for item in items:
            if item.key == f"day_{day}":
                week_data.append((day, item.value))
                break
    
    if not week_data:
        return f"No check-ins found for week {week}"
    
    result = [f"Week {week} Progress Summary (Days {start_day}-{end_day}):"]
    for day, data in sorted(week_data):
        result.append(f"\nDay {day}:")
        for key, value in data.items():
            result.append(f"  {key}: {value}")
    
    return "\n".join(result)

@tool
def save_challenge_plan(user_id: str, plan_data: dict) -> str:
    """Save the 30-day challenge plan to memory.
    
    Args:
        user_id: The user's unique identifier
        plan_data: Dictionary containing the challenge plan details
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "challenge_plan")
    memory_store.put(namespace, "plan", plan_data)
    return f"Saved 30-day challenge plan for {user_id}"

@tool
def get_challenge_plan(user_id: str) -> str:
    """Retrieve the user's 30-day challenge plan.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        Formatted challenge plan or "Not found" message
    """
    namespace = (user_id, "challenge_plan")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No challenge plan found for {user_id}"
    
    for item in items:
        if item.key == "plan":
            plan = item.value
            result = ["30-Day Challenge Plan:"]
            for key, value in plan.items():
                result.append(f"  {key}: {value}")
            return "\n".join(result)
    
    return f"No challenge plan found for {user_id}"

# Combine all tools
wellness_challenge_tools = [
    # Planning
    write_todos,
    update_todo,
    list_todos,
    # Long-term Memory
    get_user_profile,
    save_user_preference,
    # Challenge-specific tools
    save_daily_checkin,
    get_daily_checkin,
    get_weekly_progress,
    save_challenge_plan,
    get_challenge_plan,
    # Skills
    load_skill,
]

print("Additional tools defined!")


# Step 3: Build the main coordinator agent
# Create the 30-Day Wellness Challenge Coordinator
wellness_challenge_coordinator = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=wellness_challenge_tools,
    backend=activity_two_backend,  # All file ops go to workspace
    subagents=[challenge_planner, progress_analyst],
    system_prompt="""You are a 30-Day Wellness Challenge Coordinator that helps users create and complete personalized wellness challenges.

## Your Role
- Create personalized 30-day wellness challenge plans
- Track daily progress through check-ins
- Analyze weekly progress and adapt recommendations
- Generate weekly summary reports
- Remember user preferences and adapt plans accordingly

## Workflow for Creating a Challenge
1. **Get User Context**: Retrieve user profile to understand goals, preferences, and constraints
2. **Planning**: Create a todo list for challenge creation tasks
3. **Delegation**: Use challenge-planner to design the 30-day program
4. **Storage**: Save the plan to memory and as a file
5. **Documentation**: Create a comprehensive plan document in the workspace

## Workflow for Daily Check-ins
1. **Record Check-in**: Save daily progress data (energy, completed activities, notes)
2. **Analysis**: Use progress-analyst to review patterns and trends
3. **Adaptation**: Adjust recommendations if needed based on progress
4. **Encouragement**: Provide supportive feedback

## Workflow for Weekly Summaries
1. **Data Collection**: Gather all check-ins for the week
2. **Analysis**: Use progress-analyst to identify trends and insights
3. **Report Generation**: Create a comprehensive weekly summary report
4. **Adaptation**: Suggest plan adjustments if needed
5. **Storage**: Save weekly report to workspace

## Important Guidelines
- Always check user profile first for personalization
- Respect medical conditions and physical limitations
- Make challenges progressive but achievable
- Provide clear, actionable daily activities
- Celebrate progress and provide encouragement
- Adapt plans based on user feedback and progress
- Save all plans and reports to files for user reference"""
)

print("30-Day Wellness Challenge Coordinator created with all 4 Deep Agent elements!")


Activity #2 workspace configured: d:\SOFTWARES\CodeRepo\AIE9\07_Deep_Agents\workspace\activity_two
Subagent configurations defined!
Additional tools defined!
30-Day Wellness Challenge Coordinator created with all 4 Deep Agent elements!


In [15]:
# Step 4: Test with a user creating their 30-day challenge
TODO_STORE.clear()

# Test creating a 30-day challenge
print("=" * 70)
print("TEST 1: Creating a 30-Day Wellness Challenge")
print("=" * 70)

result = wellness_challenge_coordinator.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like to start a 30-day wellness challenge.

My goals are:
- Build a consistent morning exercise routine (I can do 20-30 minutes)
- Improve my sleep quality (currently getting 6 hours, want 7-8)
- Reduce work stress through daily mindfulness practices

Please create a personalized 30-day plan that:
1. Starts gentle and builds progressively
2. Includes daily activities for exercise, sleep, and stress management
3. Has weekly milestones
4. Is saved as a detailed markdown file I can reference daily

Remember I'm vegetarian and have mild anxiety, so keep exercises gentle and stress-reducing."""
    }]
})

print("\nCoordinator Response:")
print(result["messages"][-1].content)

# Check todos
print("\n" + "=" * 70)
print("TODO STATUS")
print("=" * 70)
print(list_todos.invoke({}))

# Check generated files
print("\n" + "=" * 70)
print("GENERATED FILES")
print("=" * 70)
for f in sorted(activity_two_workspace.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

# Step 5: Simulate a daily check-in and adaptation
print("\n\n" + "=" * 70)
print("TEST 2: Daily Check-in and Adaptation")
print("=" * 70)

# Simulate a check-in after a few days
result2 = wellness_challenge_coordinator.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! It's day 5 of my challenge. Here's my check-in:

Day 5 Progress:
- Energy level: 6/10 (feeling a bit tired)
- Completed activities: morning walk (20 min), evening meditation (10 min)
- Sleep: 6.5 hours (slept at 11:30pm, woke up at 6am)
- Notes: The morning walk felt good, but I'm struggling with the evening routine. Work stress is still high. I think the challenge might be a bit too ambitious for my current schedule.

Can you:
1. Record this check-in
2. Analyze my progress so far
3. Suggest any adaptations to make the challenge more sustainable
4. Create a brief progress report"""
    }]
})

print("\nCoordinator Response:")
print(result2["messages"][-1].content)

# Check if weekly summary was created
print("\n" + "=" * 70)
print("UPDATED FILES")
print("=" * 70)
for f in sorted(activity_two_workspace.iterdir()):
    if f.is_file() and ("progress" in f.name.lower() or "check" in f.name.lower() or "week" in f.name.lower()):
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")

# Test weekly summary generation
print("\n\n" + "=" * 70)
print("TEST 3: Weekly Summary Generation")
print("=" * 70)

result3 = wellness_challenge_coordinator.invoke({
    "messages": [{
        "role": "user",
        "content": """I've completed week 1 of my challenge! Can you create a comprehensive weekly summary report that:
1. Reviews all my progress from days 1-7
2. Highlights what went well
3. Identifies areas for improvement
4. Suggests adjustments for week 2
5. Saves it as a markdown file"""
    }]
})

print("\nCoordinator Response:")
print(result3["messages"][-1].content)

print("\n" + "=" * 70)
print("FINAL WORKSPACE CONTENTS")
print("=" * 70)
for f in sorted(activity_two_workspace.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

TEST 1: Creating a 30-Day Wellness Challenge

Coordinator Response:
## 🎉 Your 30-Day Wellness Challenge is Ready!

Alex, I've created a comprehensive, personalized wellness challenge plan that addresses all your goals while respecting your vegetarian lifestyle and anxiety considerations. Here's what I've prepared for you:

### ✨ What's Been Created:

1. **Complete 30-Day Plan** - Saved to your profile and as a detailed markdown file
2. **Progressive Structure** - Starting gentle (15-20 min) and building to full routines (30 min)
3. **Three Focus Areas**: 
   - 🏃‍♀️ Morning exercise routines (yoga, walking, gentle strength)
   - 😴 Sleep improvement strategies (from 6 to 7-8 hours)
   - 🧘 Daily stress management (breathing, meditation, mindfulness)

### 📋 Your Weekly Milestones:
- **Week 1**: Foundation building with gentle routines
- **Week 2**: Consistency strengthening 
- **Week 3**: Practice expansion and deepening
- **Week 4**: Mastery and long-term sustainability

### 🎯 Special Fea

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)